# 1. Data Preparation
Load `randomdata_annotated.csv`, handle missing values (KNN Imputer), engineer features, apply **temporal split** (no data leakage), and save train/test sets for model training.

> **Temporal split**: train on terms 2015-2020, test on terms 2021-2023. Terms 2024-2025 are excluded entirely (see 1.2b) because they have no forward term yet, so `at_risk` cannot be genuinely 1 for those years — including them would corrupt the test set with rows that can only ever be labeled 0.
> A random split would let the model train on Fall 2023 data and test on Fall 2015 data, leaking future patterns into training and inflating AUC.


In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
from pathlib import Path
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

DATA_PATH  = Path('randomdata_annotated.csv')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# Train: 2015-2020 | Test: 2021-2023 | Excluded: 2024-2025 (no forward term yet,
# so at_risk cannot be genuinely 1 for those years -- see 1.2b)
TRAIN_END_YEAR = 2021   # train: term_year < TRAIN_END_YEAR
TEST_END_YEAR  = 2024   # test:  TRAIN_END_YEAR <= term_year < TEST_END_YEAR
RANDOM_STATE = 42
print("Libraries loaded.")


Libraries loaded.


## 1.1 Load and inspect the raw annotated dataset

In [2]:
df = pd.read_csv(DATA_PATH, dtype=str)
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head(3)


Shape: (265668, 19)
Columns: ['student_id', 'u_g', 'gender', 'school', 'degtype', 'firstgen', 'pell', 'ethnicmultirace', 'citizen', 'creditenr', 'year', 'semester', 'academic_state', 'regstat', 'accumgpa', 'term_code', 'term_type', 'term_year', 'at_risk']


,student_id,u_g,gender,school,degtype,firstgen,pell,ethnicmultirace,citizen,creditenr,year,semester,academic_state,regstat,accumgpa,term_code,term_type,term_year,at_risk
0,STU06633,D,Female,CC,PHD,N,N,26,0,6,2015,Spring,Graduated,4,3.0,201510,Spring,2015,0.0
1,STU10115,D,Male,SL,PHD,N,Unknown,20,3,14,2015,Spring,Maintaining Registration,4,3.4,201510,Spring,2015,1.0
2,STU10810,D,Female,AD,PHD,N,N,25,1,6,2015,Spring,Enrolled,4,2.87,201510,Spring,2015,0.0


In [3]:
# at_risk = 1 (did NOT return next term), 0 (returned), NaN (no ground truth)
print("at_risk value counts (including NaN):")
print(df['at_risk'].value_counts(dropna=False))
print()
print("Missing values per column:")
print(df.isnull().sum())


at_risk value counts (including NaN):
at_risk
0.0    229851
1.0     20275
NaN     15542
Name: count, dtype: int64

Missing values per column:
student_id             0
u_g                    0
gender                 0
school                 0
degtype                0
firstgen               0
pell                   0
ethnicmultirace        0
citizen                0
creditenr              0
year                   0
semester               0
academic_state         0
regstat                0
accumgpa               0
term_code              0
term_type              0
term_year              0
at_risk            15542
dtype: int64


## 1.2 Clean: drop no-ground-truth rows, graduates, and unusable years
- Drop rows where `at_risk` is NaN (most-recent-term rows -- no next term to check)
- Drop rows where `academic_state == 'Graduated'` (successful completers, not dropout)
- Drop rows where `term_year >= 2024` -- 2024 and 2025 have no forward term yet, so every
  row in those years is necessarily at_risk=0 (never 1). Keeping them would corrupt the
  target distribution and bias the test set toward "returned".


In [4]:
# Cast target and numeric columns
df['at_risk']   = pd.to_numeric(df['at_risk'],  errors='coerce')
df['accumgpa']  = pd.to_numeric(df['accumgpa'],  errors='coerce')
df['creditenr'] = pd.to_numeric(df['creditenr'], errors='coerce')
df['term_year'] = pd.to_numeric(df['term_year'], errors='coerce')
df['regstat']   = pd.to_numeric(df['regstat'],   errors='coerce')

# Drop no-ground-truth rows
df = df[df['at_risk'].notna()].copy()
df['at_risk'] = df['at_risk'].astype(int)

# Drop graduates -- they are successful completers, not at-risk
df = df[df['academic_state'] != 'Graduated'].copy()

# Drop 2024+ -- no forward term exists yet, so at_risk is structurally always 0
# for these years (never a genuine dropout signal). Including them would corrupt
# the target distribution and bias any test split toward "returned".
df = df[df['term_year'] < TEST_END_YEAR].copy()

print(f"Rows after cleaning: {len(df):,}")
print()
print(df['at_risk'].value_counts(normalize=True).round(4))


Rows after cleaning: 189,619

at_risk
0    0.8931
1    0.1069
Name: proportion, dtype: float64


## 1.3 Missing value handling
- `pell` and `firstgen` blanks → explicit `'Unknown'` category (do NOT coerce to N — that would understate eligible/first-gen populations)
- Numeric fields (`accumgpa`, `creditenr`) → **KNN Imputer** using 5 nearest neighbors on the feature space, which is more informative than median for student records where GPA and credits are correlated with level and school

In [5]:
# Pell and firstgen: blank/NaN -> 'Unknown'
df['pell']     = df['pell'].fillna('').replace('', 'Unknown')
df['firstgen'] = df['firstgen'].fillna('').replace('', 'Unknown')

print("pell values:", df['pell'].unique())
print("firstgen values:", df['firstgen'].unique())
print(f"\naccumgpa nulls: {df['accumgpa'].isna().sum():,}")
print(f"creditenr nulls: {df['creditenr'].isna().sum():,}")


pell values: ['Unknown' 'N' 'Y']
firstgen values: ['N' 'Unknown' 'Y']

accumgpa nulls: 0
creditenr nulls: 0


## 1.4 Feature Engineering
- **Grouping for episode features**: use `(student_id, u_g, degtype, school)` — adding `school` prevents conflating different programs if a student changes college within the same degree type
- **`terms_enrolled_so_far`**: episode seniority (persistence signal)
- **`gpa_change`**: GPA momentum from prior term
- **`term_code` excluded from features**: used for temporal splitting only, not as a predictor (would encode time ordering directly into the model)
- **`term_year` excluded**: removing to avoid time-based leakage in the feature space; the temporal split already handles this at the data level

In [6]:
# Sort for proper temporal ordering within each episode
df = df.sort_values(['student_id', 'u_g', 'degtype', 'school', 'term_code'])

# Episode group key — includes school to handle school transfers
EPISODE_KEY = ['student_id', 'u_g', 'degtype', 'school']

# terms_enrolled_so_far: how many terms in this episode so far (1-indexed)
df['terms_enrolled_so_far'] = df.groupby(EPISODE_KEY).cumcount() + 1

# gpa_change: GPA momentum (0 for first term of episode)
df['prior_gpa'] = df.groupby(EPISODE_KEY)['accumgpa'].shift(1)
df['gpa_change'] = (df['accumgpa'] - df['prior_gpa']).fillna(0)

print("Derived features added: terms_enrolled_so_far, gpa_change")
print(df[['student_id','u_g','degtype','school','term_code',
          'terms_enrolled_so_far','accumgpa','gpa_change']].head(8))


Derived features added: terms_enrolled_so_far, gpa_change


   student_id u_g degtype school term_code  terms_enrolled_so_far  accumgpa  \
1    STU10115   D     PHD     SL    201510                      1      3.40   
2    STU10810   D     PHD     AD    201510                      1      2.87   
5    STU13900   D     PHD     SL    201510                      1      2.90   
8    STU15177   D     PHD     AD    201510                      1      3.16   
10   STU15290   D     PHD     SM    201510                      1      3.33   
11   STU15290   D     PHD     SM    201590                      2      3.25   
12   STU15290   D     PHD     SM    201610                      3      3.21   
14   STU15524   D     PHD     AD    201510                      1      3.19   

    gpa_change  
1         0.00  
2         0.00  
5         0.00  
8         0.00  
10        0.00  
11       -0.08  
12       -0.04  
14        0.00  


In [7]:
# Encode categorical features
pell_map    = {'Y': 1.0, 'N': 0.0, 'Unknown': 0.5}
firstgen_map = {'Y': 1.0, 'N': 0.0, 'Unknown': 0.5}

df['pell_encoded']        = df['pell'].map(pell_map).fillna(0.5)
df['firstgen_encoded']    = df['firstgen'].map(firstgen_map).fillna(0.5)
df['gender_encoded']      = (df['gender'] == 'Female').astype(int)
df['term_season_encoded'] = (df['term_type'] == 'Spring').astype(int)
df['regstat_encoded']     = df['regstat'].fillna(4).astype(float)
df['citizen_encoded']     = pd.to_numeric(df['citizen'], errors='coerce').fillna(4)
df['ethnicity_encoded']   = pd.to_numeric(df['ethnicmultirace'], errors='coerce').fillna(0)

# One-hot encode school, u_g, degtype
school_dummies  = pd.get_dummies(df['school'],   prefix='school',  dtype=int)
ug_dummies      = pd.get_dummies(df['u_g'],      prefix='ug',      dtype=int)
degtype_dummies = pd.get_dummies(df['degtype'],  prefix='deg',     dtype=int)

print("Categorical encoding done.")
print(f"school dummies: {school_dummies.columns.tolist()}")
print(f"u_g dummies:    {ug_dummies.columns.tolist()}")


Categorical encoding done.
school dummies: ['school_AD', 'school_CC', 'school_EN', 'school_SL', 'school_SM']
u_g dummies:    ['ug_D', 'ug_G', 'ug_U']


In [8]:
# Assemble base feature matrix (NO term_year, NO term_code — temporal leakage risk)
SCALAR_FEATURES = [
    'pell_encoded', 'firstgen_encoded', 'gender_encoded',
    'term_season_encoded', 'regstat_encoded',
    'accumgpa', 'creditenr',
    'terms_enrolled_so_far', 'gpa_change',
    'citizen_encoded', 'ethnicity_encoded',
]

X = pd.concat([
    df[SCALAR_FEATURES].reset_index(drop=True),
    school_dummies.reset_index(drop=True),
    ug_dummies.reset_index(drop=True),
    degtype_dummies.reset_index(drop=True),
], axis=1)

y          = df['at_risk'].reset_index(drop=True)
term_years = df['term_year'].reset_index(drop=True)

print(f"Feature matrix shape: {X.shape}")
print(f"Features: {X.columns.tolist()}")


Feature matrix shape: (189619, 30)
Features: ['pell_encoded', 'firstgen_encoded', 'gender_encoded', 'term_season_encoded', 'regstat_encoded', 'accumgpa', 'creditenr', 'terms_enrolled_so_far', 'gpa_change', 'citizen_encoded', 'ethnicity_encoded', 'school_AD', 'school_CC', 'school_EN', 'school_SL', 'school_SM', 'ug_D', 'ug_G', 'ug_U', 'deg_BA', 'deg_BAR', 'deg_BET', 'deg_BS', 'deg_CRT', 'deg_G', 'deg_MAR', 'deg_MBA', 'deg_MS', 'deg_PHD', 'deg_U']


In [9]:
# KNN Imputation for accumgpa and creditenr (numeric missing values)
# KNN uses the 5 nearest neighbors in feature space — more informative
# than median for student records where GPA correlates with level and school
numeric_impute_cols = ['accumgpa', 'creditenr']

print(f"Missing before KNN imputation:")
print(X[numeric_impute_cols].isna().sum())

# Fit imputer only on non-NaN rows (it handles NaN internally)
knn_imputer = KNNImputer(n_neighbors=5)
X[numeric_impute_cols] = knn_imputer.fit_transform(X[numeric_impute_cols])

print(f"\nMissing after KNN imputation:")
print(X[numeric_impute_cols].isna().sum())


Missing before KNN imputation:
accumgpa     0
creditenr    0
dtype: int64

Missing after KNN imputation:
accumgpa     0
creditenr    0
dtype: int64


## 1.5 Temporal Split
**Train**: term_year < 2021 (2015-2020)  
**Test**: 2021 <= term_year < 2024 (2021-2023)  
**Excluded**: term_year >= 2024 (2024-2025) -- already dropped in 1.2, no genuine dropout signal available

This ensures the model is evaluated on genuinely unseen future students, matching how the model would be deployed in production, without contaminating the test set with years that structurally cannot contain a positive label.


In [10]:
train_mask = term_years < TRAIN_END_YEAR
test_mask  = (term_years >= TRAIN_END_YEAR) & (term_years < TEST_END_YEAR)

X_train, y_train = X[train_mask].reset_index(drop=True), y[train_mask].reset_index(drop=True)
X_test,  y_test  = X[test_mask].reset_index(drop=True),  y[test_mask].reset_index(drop=True)

print(f"Temporal split (train < {TRAIN_END_YEAR}, test {TRAIN_END_YEAR}-{TEST_END_YEAR - 1}):")
print(f"  Train: {len(X_train):,} rows | years: {sorted(term_years[train_mask].unique().tolist())}")
print(f"  Test:  {len(X_test):,}  rows | years: {sorted(term_years[test_mask].unique().tolist())}")
print(f"  Train positive rate (at-risk): {y_train.mean():.4f}")
print(f"  Test  positive rate (at-risk): {y_test.mean():.4f}")


Temporal split (train < 2021, test 2021-2023):
  Train: 124,463 rows | years: [2015, 2016, 2017, 2018, 2019, 2020]
  Test:  65,156  rows | years: [2021, 2022, 2023]
  Train positive rate (at-risk): 0.1072
  Test  positive rate (at-risk): 0.1065


In [11]:
# Save all artifacts
joblib.dump(X_train,              OUTPUT_DIR / 'X_train.pkl')
joblib.dump(X_test,               OUTPUT_DIR / 'X_test.pkl')
joblib.dump(y_train,              OUTPUT_DIR / 'y_train.pkl')
joblib.dump(y_test,               OUTPUT_DIR / 'y_test.pkl')
joblib.dump(X.columns.tolist(),   OUTPUT_DIR / 'feature_names.pkl')
joblib.dump(knn_imputer,          OUTPUT_DIR / 'knn_imputer.pkl')

print("Saved to outputs/: X_train, X_test, y_train, y_test, feature_names, knn_imputer")
print("\nNext: run 02_train.ipynb")


Saved to outputs/: X_train, X_test, y_train, y_test, feature_names, knn_imputer



Next: run 02_train.ipynb
